# D.R.E.A.M.S — Maritime YOLOv8n Training (Colab)

Train the maritime detector on a free Colab GPU, export to ONNX, and download it for the Pi 5.
Set **Runtime → Change runtime type → GPU (T4)** first. See `ml/README.md` for the full pipeline.

In [ ]:
# 1. Setup
!pip -q install ultralytics onnx onnxruntime roboflow albumentations
from google.colab import drive
drive.mount('/content/drive')  # for saving checkpoints/weights

In [ ]:
# 2. Clone the repo
!git clone https://github.com/ka1e27/D.R.E.A.M.S.git
%cd D.R.E.A.M.S

In [ ]:
# 3. Download + prepare data
import os
os.environ['ROBOFLOW_API_KEY'] = 'YOUR_KEY_HERE'  # https://roboflow.com → Settings → API
!python ml/data/download_datasets.py
!python ml/data/prepare_dataset.py
# Optional maritime augmentation (doubles the train set):
# !python ml/data/augment.py

In [ ]:
# 4. Train (T4 handles batch 32; drop to 16 if OOM)
!python ml/train.py --epochs 100 --batch 32

In [ ]:
# 5. Validate + benchmark
!python ml/validate.py

In [ ]:
# 6. Export to ONNX and back up the weights to Drive
!python ml/export_onnx.py
!cp pi5/cv/models/maritime_yolov8n.onnx /content/drive/MyDrive/
!cp ml/runs/maritime_v1/weights/best.pt /content/drive/MyDrive/

In [ ]:
# 7. Download the ONNX to your local machine (then scp it to the Pi)
from google.colab import files
files.download('pi5/cv/models/maritime_yolov8n.onnx')